In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.master("local[6]").appName("Local Spark") .config('spark.ui.port', '4040') .getOrCreate()
sc = spark.sparkContext

sc


<SparkContext master=local[6] appName=Local Spark>

In [3]:
# Using a small sized part of the dataset, for now
dataset_path = "/home/jovyan/work/dataset/output/csv/*.csv"
#laptimes_path

rddTelemetry = sc.textFile(dataset_path)
print(f"Number of partitions: {rddTelemetry.getNumPartitions()}")


Number of partitions: 2250


In [4]:
def parseTelemetryRow(row):
    splitted = row.split(",")

    (year, event, session_type, driver_name, lap_number, distance_driver_ahead, driver_ahead, acc_x, acc_y,
     acc_z, brake, distance, drs, gear, rel_distance, rpm, speed, throttle, time, x, y, z) = [x for x in splitted]

    return ((event, session_type, driver_name), (float(acc_y), int(brake), float(rpm), float(speed), float(throttle), rel_distance))

In [5]:
# parsing each row by creating key-value rows
columnNames = rddTelemetry.take(1)
print(columnNames)
rddTelemetryKV = (rddTelemetry \
    .filter(lambda x: x != columnNames[0])
    .map(lambda x: parseTelemetryRow(x))
    .filter(lambda x: x[1][5] != 'None'))

rddTelemetryKV = rddTelemetryKV.map(lambda x: (x[0], (x[1][0],x[1][1],x[1][2],x[1][3],x[1][4])))


['year,event,sessionType,driverName,lapNumber,DistanceToDriverAhead,DriverAhead,acc_x,acc_y,acc_z,brake,distance,drs,gear,rel_distance,rpm,speed,throttle,time,x,y,z']


In [6]:
#print(f"Number of rows: {rddTelemetryKV.count()}")

In [7]:
# aggregating by key based on 95th percentile for lateral acceleration, average on braking, engine rpms while accelerating

# sequencing function on (acc_y, brake, rpm, speed, throttle) to calc average and give as a result (max_acc_y, sum_brake, sum_rpm, rows_throttle_speed_threshold, total_rows)
seqFunc = (
    lambda x, y:
    (x[0] if x[0] > y[0] else y[0],
     x[1] + y[1],
     (x[2] + y[2]) if (y[3] > 120 and y[4] > 35) else x[2],
     (x[3] + 1) if (y[3] > 120 and y[4] > 35) else x[3],
     x[4] + 1)
)

#combining function between partitions
combFunc = (
    lambda x, y:
    (x[0] if x[0] > y[0] else y[0],
     x[1] + y[1],
     x[2] + y[2],
     x[3] + y[3],
     x[4] + y[4])
)

#mapping the values
mapFunc = (
    lambda x:
    (x[0],
     x[1]/x[4],
     x[2]/x[3] if x[3] > 0 else x[1])
)

#accumulator is (max_acc_y, sum_brake, sum_rpm, rows_throttle_speed_threshold, total_rows)
rddDrivingStyleParams = rddTelemetryKV\
    .aggregateByKey((-30.0, 0, 0, 0, 0), seqFunc, combFunc)\
    .mapValues(mapFunc)

# (rddTelemetryKV\
#     .map(lambda x: (x[0], (x[1][1], 1)))\
#     .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))\
#     .mapValues(lambda x: x[0]/x[1])\
#     .collect())

In [8]:
# assigning a "style" label to each row based on some parameters that define the style

# we assume that above average is considered agressive in at least 2 of three categories such as max lateral acceleration and high rpms
# one category higher than the average is considered balanced
# the rest is conservative
# as averages we take 30 m/s^2 as the high threshold, braking percentage 20% and rpms around 10500
acc_y_mean_threshold = rddDrivingStyleParams.map(lambda x: x[1][0]).mean()
brake_mean_threshold = rddDrivingStyleParams.map(lambda x: x[1][1]).mean()
rpms_mean_threshold = rddDrivingStyleParams.map(lambda x: x[1][2]).mean()

def assigningFunc(x):
    score = 0
    if x[0] > acc_y_mean_threshold: score+=1
    if x[1] > brake_mean_threshold: score+=1
    if x[2] > rpms_mean_threshold: score+=1
    if score >= 2: return (x, 'AGRESSIVE')
    elif (score == 1): return (x, 'BALANCED')
    else: return (x, 'CONSERVATIVE')


rddTelemetryWStyle = rddDrivingStyleParams.mapValues(assigningFunc).map(lambda x: (x[0], x[1][1]))#.sortBy(lambda x: x[1], ascending=True).collect()

print(rddTelemetryWStyle)

PythonRDD[10] at RDD at PythonRDD.scala:53


In [9]:
print(rddTelemetryWStyle.collect())

[(('Singapore Grand Prix', 'Qualifying', 'GAS'), 'BALANCED'), (('British Grand Prix', 'Practice 2', 'HUL'), 'CONSERVATIVE'), (('Dutch Grand Prix', 'Qualifying', 'LAW'), 'AGRESSIVE'), (('British Grand Prix', 'Practice 2', 'ALO'), 'AGRESSIVE'), (('Hungarian Grand Prix', 'Practice 3', 'HAM'), 'BALANCED'), (('Monaco Grand Prix', 'Practice 3', 'RUS'), 'AGRESSIVE'), (('Azerbaijan Grand Prix', 'Qualifying', 'ANT'), 'AGRESSIVE'), (('Austrian Grand Prix', 'Race', 'RUS'), 'BALANCED'), (('Hungarian Grand Prix', 'Practice 1', 'TSU'), 'CONSERVATIVE'), (('São Paulo Grand Prix', 'Race', 'BEA'), 'BALANCED'), (('Chinese Grand Prix', 'Qualifying', 'HUL'), 'AGRESSIVE'), (('Australian Grand Prix', 'Practice 3', 'HUL'), 'AGRESSIVE'), (('Japanese Grand Prix', 'Practice 1', 'TSU'), 'BALANCED'), (('Canadian Grand Prix', 'Practice 1', 'LAW'), 'AGRESSIVE'), (('Abu Dhabi Grand Prix', 'Practice 1', 'BEA'), 'AGRESSIVE'), (('São Paulo Grand Prix', 'Qualifying', 'PIA'), 'AGRESSIVE'), (('Australian Grand Prix', 'Prac

In [10]:
#loading the second dataset
laptimes_path = "/home/jovyan/work/dataset/output/laptimes/*.csv"
rddLapTimes = sc.textFile(laptimes_path)
print(f"Number of partitions: {rddLapTimes.getNumPartitions()}")

Number of partitions: 77


In [11]:
def parseLaptimesRow(row):
    splitted = row.split(",")
    print(splitted)
    (event,sessionType,totalLaps,team,driverCode,driverNumber,lap,lapTime,isPersonalBest,position,pitOut,tyreCompound,tyreAgeLaps,stintNumber,sector1Time,sector2Time,sector3Time,speedTrapIntermediate1,speedTrapIntermediate2,speedFinishLine,speedStraight,airTemperature,humidity,atmPressure,isRaining,trackTemperature,windDirection,windSpeed) = [x for x in splitted]

    return ((event, sessionType, driverCode), (float(lapTime) if lapTime != 'None' else 0.0, tyreCompound if tyreCompound != 'None' else '', int(tyreAgeLaps) if tyreAgeLaps != '' else 0, int(lap), int(totalLaps), pitOut))

In [12]:
#parsing and cleaning the second dataset
columnNames = rddLapTimes.take(1)
print(columnNames)
rddLapTimesKV = (rddLapTimes \
                  .filter(lambda x: x != columnNames[0])
                  .map(lambda x: parseLaptimesRow(x))\
                  .filter(lambda x: x[1][0] != 0.0 and x[1][1] != '' and x[1][2] != 0))

['event,sessionType,totalLaps,team,driverCode,driverNumber,lap,lapTime,isPersonalBest,position,pitOut,tyreCompound,tyreAgeLaps,stintNumber,sector1Time,sector2Time,sector3Time,speedTrapIntermediate1,speedTrapIntermediate2,speedFinishLine,speedStraight,airTemperature,humidity,atmPressure,isRaining,trackTemperature,windDirection,windSpeed']


In [13]:
#joining the two datasets hopefully not breeaking anything
joinedRDD = rddTelemetryWStyle.join(rddLapTimesKV)

joinedRDD.top(50)


[(('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (141.589, 'MEDIUM', 4, 4, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (138.286, 'MEDIUM', 3, 9, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (137.497, 'MEDIUM', 3, 3, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (130.416, 'SOFT', 3, 12, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (116.258, 'MEDIUM', 6, 6, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (94.14, 'MEDIUM', 2, 2, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (93.363, 'MEDIUM', 5, 5, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRESSIVE', (93.163, 'MEDIUM', 2, 8, 12, 'None'))),
 (('United States Grand Prix', 'Sprint Qualifying', 'VER'),
  ('AGRES

In [72]:
def assignTyreWearFunc(x):
    lap_on_tyre = x[1][2]
    compound    = x[1][1]
    lap_number  = x[1][3]

    if lap_on_tyre == 1:
        return (x, 'DROP')

    if lap_number <= 15:        # First stint
        if compound == 'HARD':
            optimal_max = 16
        elif compound == 'MEDIUM':
            optimal_max = 12
        else:
            optimal_max = 6
    else:                       # Normal stints
        if compound == 'SOFT':
            optimal_max = 5
        elif compound == 'MEDIUM':
            optimal_max = 12
        elif compound == 'HARD':
            optimal_max = 18
        elif compound == 'INTERMEDIATE':
            optimal_max = 7
        elif compound == 'WET':
            optimal_max = 5
        else:
            optimal_max = 6

    return (x, 'OPTIMAL' if lap_on_tyre <= optimal_max else 'DEGRADED')

In [73]:
joinedRddLabeled = (joinedRDD.mapValues(assignTyreWearFunc) \
                    .filter(lambda x: x[1][1] != 'DROP')\
                    .mapValues(lambda x: (x[0][0], x[0][1][1], x[1], x[0][1][0], x[0][1][3], x[0][1][4], x[0][1][5], 1))\
                    .mapValues(lambda x: (x[0], x[1], x[2], x[3] + ((100 * (1 - x[4]/x[5])) * 0.03), x[4], x[5], x[6], x[7]))\
                    .filter(lambda x: x[0][1] == 'Race' and x[1][6] == 'None' and x[1][4] > 3))

In [74]:
finalRdd = joinedRddLabeled \
    .map(lambda x: ((x[1][0], x[1][1], x[1][2]), x[1][3])) \
    .groupByKey() \
    .mapValues(lambda times: sorted(list(times))) \
    .mapValues(lambda t: (t[len(t)//2], len(t), sum(t)/len(t)))

In [54]:
finalRdd.top(50)


[(('CONSERVATIVE', 'SOFT', 'OPTIMAL'),
  {'median': 83.9770909090909, 'count': 66, 'mean': 88.65189072545644}),
 (('CONSERVATIVE', 'SOFT', 'DEGRADED'),
  {'median': 84.78318181818182, 'count': 424, 'mean': 88.44138063375779}),
 (('CONSERVATIVE', 'MEDIUM', 'OPTIMAL'),
  {'median': 86.41765217391304, 'count': 302, 'mean': 92.50364240346501}),
 (('CONSERVATIVE', 'MEDIUM', 'DEGRADED'),
  {'median': 84.427, 'count': 558, 'mean': 88.83316509848207}),
 (('CONSERVATIVE', 'INTERMEDIATE', 'OPTIMAL'),
  {'median': 125.64957894736843, 'count': 92, 'mean': 123.24387641418605}),
 (('CONSERVATIVE', 'INTERMEDIATE', 'DEGRADED'),
  {'median': 95.84863157894738, 'count': 465, 'mean': 100.2181810380939}),
 (('CONSERVATIVE', 'HARD', 'OPTIMAL'),
  {'median': 85.98873913043478, 'count': 295, 'mean': 94.08092540075226}),
 (('CONSERVATIVE', 'HARD', 'DEGRADED'),
  {'median': 84.6375652173913, 'count': 487, 'mean': 87.90206088913145}),
 (('BALANCED', 'SOFT', 'OPTIMAL'),
  {'median': 83.54357746478874, 'count': 2

In [75]:
# Calculate pace drop-off = median(DEGRADED) - median(OPTIMAL)
dropoff_rdd = finalRdd \
    .map(lambda x:
         ((x[0][0], x[0][1]),                    # (style, compound)
          (x[0][2], x[1][0], x[1][1]))           # (tyre_state, median, count)
         ) \
    .groupByKey() \
    .mapValues(list)

def calculate_dropoff(items):
    data = {state: (med, cnt) for state, med, cnt in items}

    opt_med = data.get('OPTIMAL', (None, 0))[0]
    deg_med = data.get('DEGRADED', (None, 0))[0]
    opt_cnt = data.get('OPTIMAL', (0, 0))[1]
    deg_cnt = data.get('DEGRADED', (0, 0))[1]

    drop = (deg_med - opt_med) if opt_med is not None and deg_med is not None else None

    return {
        'optimal_median': round(opt_med, 3) if opt_med else None,
        'degraded_median': round(deg_med, 3) if deg_med else None,
        'pace_dropoff_s': round(drop, 3) if drop is not None else None,
        'optimal_count': opt_cnt,
        'degraded_count': deg_cnt
    }

result = dropoff_rdd.mapValues(calculate_dropoff).collect()

# Pretty output
print(f"{'Style':12} {'Compound':10} {'Drop-off (s)':>12}   Opt Median (count)   Deg Median (count)")
print("-" * 80)
for (style, compound), stats in sorted(result):
    if stats['pace_dropoff_s'] is not None:
        print(f"{style:12} {compound:10} {stats['pace_dropoff_s']:12.3f}   "
              f"{stats['optimal_median']:8.3f} ({stats['optimal_count']:4})   "
              f"{stats['degraded_median']:8.3f} ({stats['degraded_count']:4})")

Style        Compound   Drop-off (s)   Opt Median (count)   Deg Median (count)
--------------------------------------------------------------------------------
AGRESSIVE    HARD             -0.684     92.184 (2977)     91.500 (2329)
AGRESSIVE    INTERMEDIATE        1.394    124.464 (  71)    125.858 ( 141)
AGRESSIVE    MEDIUM            2.822     93.334 (2004)     96.156 (2167)
AGRESSIVE    SOFT              0.143     99.148 (  88)     99.291 ( 637)
BALANCED     HARD             -1.066     84.966 (2369)     83.900 (1786)
BALANCED     INTERMEDIATE      -25.459    125.366 ( 116)     99.908 ( 382)
BALANCED     MEDIUM           -1.823     85.399 (1858)     83.576 (2535)
BALANCED     SOFT              0.488     83.544 ( 201)     84.031 (1510)
CONSERVATIVE HARD             -1.362     85.654 ( 477)     84.292 ( 305)
CONSERVATIVE INTERMEDIATE      -29.912    125.625 ( 119)     95.713 ( 438)
CONSERVATIVE MEDIUM           -1.863     86.088 ( 466)     84.224 ( 394)
CONSERVATIVE SOFT              